In [1]:
import os
import glob
import shutil
import random
import sys
import json

import numpy as np
import xml.etree.ElementTree as ET
import yaml
from tqdm.contrib.concurrent import thread_map
import cv2

sys.path.append("../..")
sys.path.append("../../object-detection-v3")
sys.path.append("../../data-exploration/video_splitting")
from annotation_poc import utils as annotation_utils
from utils import yolov8_dataset_util
from splitter import create_project_file

%load_ext autoreload
%autoreload 2

In [2]:
ROOT_PTH = "/home/klymy/data/cam4_revB"
BATHCHES_RANGE = [6, 7]
SUBSETS_FILE_NAME = "train_val_holdout_subsets_batches_6_7.json"
DATASET_NAME = "batch6_7"

BASE_ANNOTATION_FOLDERS = [os.path.join(ROOT_PTH, f"batch{i}_corrected") for i in BATHCHES_RANGE]
BASE_FOLDERS = [os.path.join(ROOT_PTH, f"4.2_batch{i}") for i in BATHCHES_RANGE]
BASE_FOLDERS, BASE_ANNOTATION_FOLDERS

(['/home/klymy/data/cam4_revB/4.2_batch6',
  '/home/klymy/data/cam4_revB/4.2_batch7'],
 ['/home/klymy/data/cam4_revB/batch6_corrected',
  '/home/klymy/data/cam4_revB/batch7_corrected'])

In [3]:
with open(os.path.join(ROOT_PTH, SUBSETS_FILE_NAME), "rt") as f:
    train_val_holdout_subsets = json.load(f)

In [4]:
def cp_img(args) -> None:
    pth, obj = args
    img = cv2.imread(obj["path"])
    cv2.imwrite(os.path.join(pth, f"{obj['id']}.jpg"), img)

In [5]:
def img_pth2id(img_pth: str, prefix='', base_pth="") -> str:
    img_name = img_pth.replace(base_pth, "")
    return f'{prefix}{img_name.replace("/", "____").replace(".tif", "")}'

In [6]:
def split(objects: dict, splitting: dict) -> tuple:
    """_summary_

    Args:
        objects (dict): _description_
        splitting (dict): _description_

    Returns:
        tuple: _description_
    """
    train_ids = []
    val_ids = []
    holdout_ids = []
    for i in objects:
        if i["path"] in splitting.get("train", []):
            train_ids.append(i["id"])
        elif i["path"] in splitting.get("val", []):
            val_ids.append(i["id"])
        elif i["path"] in splitting.get("holdout", []):
            holdout_ids.append(i["id"])
        else:
            raise ValueError(f"Unknown id {i['id']}")
    
    return train_ids, val_ids, holdout_ids

In [7]:

def create_yolo_dataset(
        objects: list, label_mapping: dict, dataset_pth: str,
        train_val_holdout_subsets: dict,
) -> None:
    """_summary_

    Args:
        objects (list): list of objects. Object is an image with annotation:
        {
            "id": str,
            "path": str,
            "objects": List[dict]: [
                {
                    "cls": int,
                    "xtl": float,
                    "ytl": float,
                    "xbr": float,
                    "ybr": float,
                    "x": float,
                    "y": float
                }
            ]
            
        }.
        label_mapping (dict): labels mapping, e.g. {0: desk, 1: chair}.
        base_imgs_pth (str): _description_
        dataset_pth (str): _description_
        val_size (float, optional): _description_. Defaults to 0.1.
        random_state (int, optional): _description_. Defaults to 42.
    """
    train_ids, val_ids, holdout_ids = split(objects, train_val_holdout_subsets)
    train_objects = list(filter(lambda x: x["id"] in train_ids, objects))
    val_objects = list(filter(lambda x: x["id"] in val_ids, objects))
    holdout_objects = list(filter(lambda x: x["id"] in holdout_ids, objects))

    imgs_pth = os.path.join(dataset_pth, "images")
    labels_pth = os.path.join(dataset_pth, "labels")

    shutil.rmtree(dataset_pth, ignore_errors=True)
    os.makedirs(dataset_pth)
    os.makedirs(imgs_pth)
    os.makedirs(os.path.join(imgs_pth, "train"))
    os.makedirs(os.path.join(imgs_pth, "val"))
    if len(holdout_ids) > 0:
        os.makedirs(os.path.join(imgs_pth, "holdout"))
    os.makedirs(labels_pth)
    os.makedirs(os.path.join(labels_pth, "train"))
    os.makedirs(os.path.join(labels_pth, "val"))
    if len(holdout_ids) > 0:
        os.makedirs(os.path.join(labels_pth, "holdout"))

    thread_map(yolov8_dataset_util.create_label_txt,
               zip([os.path.join(labels_pth, "train")] * len(train_ids), train_objects), desc="train labels")
    thread_map(yolov8_dataset_util.create_label_txt,
               zip([os.path.join(labels_pth, "val")] * len(val_ids), val_objects), desc="val labels")
    thread_map(yolov8_dataset_util.create_label_txt,
               zip([os.path.join(labels_pth, "holdout")] * len(holdout_ids), holdout_objects), desc="holdout labels")

    thread_map(
        cp_img,
        zip([os.path.join(imgs_pth, "train")] * len(train_ids), train_objects),
        desc="train images", max_workers=8
    )
    thread_map(
        cp_img,
        zip([os.path.join(imgs_pth, "val")] * len(val_ids), val_objects),
        desc="val images", max_workers=8
    )
    thread_map(
        cp_img,
        zip([os.path.join(imgs_pth, "holdout")] * len(holdout_ids), holdout_objects),
        desc="holdout images", max_workers=8
    )

    create_project_file(os.path.join(dataset_pth, "project.yaml"), dataset_pth, label_mapping)
    if len(holdout_ids) > 0:
        create_project_file(
            os.path.join(dataset_pth, "project_holdout.yaml"),
            dataset_pth,
            label_mapping,
            val_folder="images/holdout"
        )


In [10]:
def convert_annotation(annotations, img_paths, prefix='', base_pth=""):
    objects = []
    for img_pth in img_paths:
        img_name = img_pth.replace(base_pth, "")
        if img_name in annotations:
            objects.append({
                "id": img_pth2id(img_pth, prefix=prefix, base_pth=base_pth),
                "path": img_pth,
                "objects": [
                    {
                        "cls": 0,
                        "xtl": obj["bounding_box"][0] / annotations[img_name]["width"],
                        "ytl": obj["bounding_box"][1] / annotations[img_name]["height"],
                        "xbr": obj["bounding_box"][2] / annotations[img_name]["width"],
                        "ybr": obj["bounding_box"][3] / annotations[img_name]["height"],
                        "x": obj["points"][0][0] / annotations[img_name]["width"],
                        "y": obj["points"][0][1] / annotations[img_name]["height"]
                    } for i, obj in enumerate(annotations[img_name]["objects"])
                ]
            })
    return objects

In [8]:
img_paths = [glob.glob(os.path.join(i, "*.tif")) for i in BASE_FOLDERS]

annotations = [
    annotation_utils.parse_and_merge_annotations(os.path.join(i, f"batch{idx}_corrected.xml")) \
        for idx, i in zip(BATHCHES_RANGE, BASE_ANNOTATION_FOLDERS)
]
sum(len(i) for i in img_paths), sum(len(i) for i in annotations)

(1700, 1700)

In [11]:
objects = []
for i, j in zip(annotations, img_paths):
    objects += convert_annotation(i, j, "", ROOT_PTH + "/")

len(objects)

1700

In [12]:
create_yolo_dataset(
    objects, {0: "person"},
    os.path.join(ROOT_PTH, "yolo_dataset", DATASET_NAME),
    train_val_holdout_subsets
)

train labels: 0it [00:00, ?it/s]

val labels: 0it [00:00, ?it/s]

holdout labels: 0it [00:00, ?it/s]

train images: 0it [00:00, ?it/s]

val images: 0it [00:00, ?it/s]

holdout images: 0it [00:00, ?it/s]